# M13-N — LoRanPAC numerical audit on Kaggle (train-only)

Set **Accelerator = GPU** and **Internet = On**. Attach `zaphat206/cifar-100`. Rename the existing M13 artifact to `srq_generalization_m13_loranpac_train_only.zip.bin` without changing its bytes, place that one file in a private Kaggle Dataset, and attach it. Kaggle would extract a `.zip`; the `.bin` suffix preserves the exact raw bytes. Use **Save Version → Save & Run All** for an overnight run.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='7f8983ebf01116d1ba6659eba14d6ef71ff1030c'
WORK_DIR='/kaggle/temp/SOHO-CL'
FEATURE_CACHE_DIR='/kaggle/temp/srq_m13n_cifar_features'
OUTPUT_DIR='/kaggle/working/srq_m13n_output'
EXPORT_PATH='/kaggle/working/srq_generalization_m13n_numerical_audit.zip'
CONFIG='configs/srq_generalization_m13n_numerical_audit.json'
RUNNER='tools/srq_generalization_m13n.py'
SOURCE_NAME='srq_generalization_m13_loranpac_train_only.zip'
SOURCE_SHA='b7cc3e1993b150d829806ac8062b10a2e31ad9c533ef729ce7a806647496d28c'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Pinned checkout, dependencies, GPU, and portable source hashes.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
assert Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir()
repo=Path(WORK_DIR); repo.parent.mkdir(parents=True,exist_ok=True)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--no-checkout',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Kaggle GPU and restart the session.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m13n_numerical_audit.json':'148506006c81a0139ea8fc5a801d79b503b43bf988fce83d28dbccb8c352db71',
 'tools/srq_generalization_m13n.py':'760ea95246f7c3e85c74c431c2d0bb9117564b9ed2ebf217b67989724b741a4c',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda',
 'tests/test_srq_generalization_m13n.py':'c142cd3bbba19fdf67b0a4c91ae31e819dfe349629281ef03b779feec34871c8'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| M13-N SOURCE LOCK: PASS')

In [ ]:
# Discover byte-preserved M13 artifact, raw CIFAR input, and checkpoint.
INPUT_ROOT=Path('/kaggle/input')
matches=sorted(path for path in INPUT_ROOT.rglob(SOURCE_NAME+'.bin') if path.is_file())
assert len(matches)==1,f'Attach exactly one {SOURCE_NAME}.bin; found {matches}'
assert sha_raw(matches[0])==SOURCE_SHA,(matches[0],sha_raw(matches[0]),SOURCE_SHA)
stage=Path('/kaggle/temp/srq_m13n_source'); stage.mkdir(parents=True,exist_ok=True)
source_copy=stage/SOURCE_NAME; shutil.copyfile(matches[0],source_copy); SOURCE_ARTIFACT=str(source_copy)
assert sha_raw(source_copy)==SOURCE_SHA
cifar_dirs=sorted({p.parent for p in INPUT_ROOT.rglob('meta') if p.is_file() and (p.parent/'train').is_file() and (p.parent/'test').is_file()})
assert len(cifar_dirs)==1,f'Attach zaphat206/cifar-100 exactly once; found {cifar_dirs}'
CIFAR_ROOT=str(cifar_dirs[0])
candidates=[p for p in INPUT_ROOT.rglob('model.safetensors') if p.is_file() and p.stat().st_size==CHECKPOINT_SIZE and sha_raw(p)==CHECKPOINT_SHA]
if candidates: assert len(candidates)==1; CHECKPOINT_PATH=str(candidates[0])
else:
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE and sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
print('M13 RAW BYTES + CIFAR + CHECKPOINT: PASS; source container unopened')

In [ ]:
# Focused gates, then TRAIN-only feature extraction.
subprocess.run([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m13n.py','tests/test_loranpac_analytic_frontend.py'],check=True)
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/kaggle/working/unused_m13n','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('M13-N PREFLIGHT + TRAIN CACHE: PASS; test.pt ABSENT')

In [ ]:
# Numerical-only audit. No prediction or held-out evaluation occurs.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m13-artifact',SOURCE_ARTIFACT,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M13-N START: two task-one SVDs; four fixed rank prefixes.',flush=True)
completed=subprocess.run(command); RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m13n_results.json'
assert result_path.is_file(),'M13-N failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status']); print('SUMMARY:',json.dumps(result['summary'],indent=2)); print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Paper-ready numerical table and vector figure.
import pandas as pd, matplotlib.pyplot as plt
rows=[]
for unit in result['units']:
    for variant in ('raw','qr_reorthogonalized_diagnostic'):
        block=unit['audit'][variant]; rows.append({'width':unit['width'],'budget':unit['budget_target'],'rank':unit['effective_rank'],'variant':variant,'fro':block['orthogonality']['raw_frobenius'],'fro_sqrt_r':block['orthogonality']['frobenius_over_sqrt_rank'],'spectral':block['orthogonality']['spectral_norm'],'solver_max':max(block['solver'].values())})
frame=pd.DataFrame(rows); display(frame)
fig,axes=plt.subplots(1,2,figsize=(10.5,4))
for (width,budget),part in frame.groupby(['width','budget']):
    label=f'{width//1000}k/{budget}'; axes[0].plot(part.variant,part.fro_sqrt_r,marker='o',label=label); axes[1].plot(part.variant,part.solver_max,marker='o',label=label)
axes[0].set_ylabel(r'$||U^TU-I||_F/\sqrt{r}$'); axes[1].set_ylabel('Maximum projected-solve residual')
for ax in axes: ax.set_yscale('log'); ax.grid(alpha=.25); ax.tick_params(axis='x',rotation=15); ax.legend(fontsize=6)
fig.tight_layout(); plot=Path(OUTPUT_DIR)/'m13n_numerical_audit.svg'; fig.savefig(plot,format='svg'); plt.show(); plt.close(fig)

In [ ]:
# Export compact evidence under /kaggle/working; source and caches are excluded.
import zipfile
export=Path(EXPORT_PATH)
members=[Path(OUTPUT_DIR)/'m13n_results.json',Path(OUTPUT_DIR)/'m13n_numerical_metrics.csv',Path(OUTPUT_DIR)/'m13n_numerical_audit.svg',Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M13N_PROTOCOL.md')]
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members: archive.write(path,path.name); manifest[path.name]=sha_raw(path)
    archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M13-N numerical audit','m13_status_remains':'FAIL_M13_LORANPAC_TRAIN_ONLY','files':manifest},indent=2)+'\n')
assert export.is_file() and zipfile.is_zipfile(export)
print('FINAL ARTIFACT:',export,'SHA-256:',sha_raw(export),'bytes:',export.stat().st_size)
from IPython.display import FileLink,display
display(FileLink(str(export)))
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M13N_NUMERICAL_AUDIT','Preserve artifact/output; do not relax gates.'